# QSOLVE Agriculture Track 1
## Hybrid Quantum-Classical Split-Delivery Multi-Factory Routing

**Quantum algorithm:** CUBO-QAOA  
**Platform:** qBraid Lab / Qiskit  
**Depot:** Karatina  
**Truck capacity:** 10 tonnes  
**Depot stock:** 410 tonnes  
**Total demand:** 233.700 tonnes  
**Transport rate:** KSh 16 / tonne-km  
**Maximum route time:** 8 hours

### Hybrid architecture

1. Load the frozen QSOLVE dataset.
2. Split factory demand into full 10-tonne loads and residual quantities.
3. Keep full single-factory loads classical.
4. Cluster residual quantities into multi-factory trips.
5. Optimize factory order in each multi-factory trip using **CUBO-QAOA**.
6. Compare each QAOA solution with an exact permutation benchmark.
7. Recombine all trips and validate the full logistics solution.

### Why CUBO-QAOA?

For route-position variables \(x_{i,p}\), the load on a road leg depends on which factories were visited earlier. This produces cubic binary interactions, so a **Cubic Unconstrained Binary Optimization (CUBO)** Hamiltonian preserves the load-dependent tonne-km objective better than a distance-only TSP QUBO.

For \(m\) factories, the position encoding uses \(m^2\) qubits. This notebook limits each quantum subproblem to at most 4 factories.

In [1]:
# Run once in qBraid Lab.
# Restart the kernel if requested after installation.

%pip install -q "qiskit>=2.3,<3" "qiskit-algorithms>=0.4,<0.5" "qbraid>=0.12" pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import time
import os
from itertools import combinations, permutations
from collections import defaultdict

import numpy as np
import pandas as pd

import qiskit
import qiskit_algorithms

from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals

print("Qiskit version:", qiskit.__version__)
print("Qiskit Algorithms version:", qiskit_algorithms.__version__)

Qiskit version: 2.5.2
Qiskit Algorithms version: 0.4.0


## 1. Experiment configuration

In [3]:
FAST_MODE = True

QAOA_REPS = 1
CVAR_ALPHA = 0.25
MAX_QUANTUM_FACTORIES = 4

if FAST_MODE:
    SHOTS = 2048
    MAXITER = 80
    RANDOM_SEEDS = [42]
else:
    SHOTS = 4096
    MAXITER = 200
    RANDOM_SEEDS = [7, 19, 42]

EPS = 1e-9

print("FAST_MODE:", FAST_MODE)
print("QAOA reps:", QAOA_REPS)
print("Shots:", SHOTS)
print("COBYLA maxiter:", MAXITER)
print("Seeds:", RANDOM_SEEDS)

FAST_MODE: True
QAOA reps: 1
Shots: 2048
COBYLA maxiter: 80
Seeds: [42]


## 2. Load the frozen QSOLVE dataset

The notebook first looks for `QSOLVE_Quantum_Ready_Dataset.xlsx` in the current qBraid notebook directory. If it is not present, the exact same frozen data are reconstructed from the embedded matrices.

In [4]:
DATA_FILE = "QSOLVE_Quantum_Ready_Dataset.xlsx"

DEPOT = "Karatina"

FACTORIES = [
    "Ragati",
    "Nguguini",
    "Kiangai",
    "Thunguri",
    "Kianjega",
    "Chema",
    "Mukangu",
]

FROZEN_DEMAND = {
    "Ragati": 28.380,
    "Nguguini": 73.092,
    "Kiangai": 36.489,
    "Thunguri": 37.170,
    "Kianjega": 24.522,
    "Chema": 10.848,
    "Mukangu": 23.199,
}

LABELS = [DEPOT] + FACTORIES

FROZEN_DISTANCE = np.array([
    [0,     11.82, 18.92, 20.02, 23.13, 13.11, 11.95, 10.08],
    [11.82, 0,     15.86, 29.64, 34.81, 25.51, 11.53, 21.86],
    [18.92, 15.86, 0,     19.08, 40.65, 20.23,  9.84, 24.95],
    [20.02, 29.64, 19.08, 0,     38.90,  7.71, 21.17, 21.38],
    [23.13, 34.81, 40.65, 38.90, 0,     32.26, 33.69, 24.62],
    [13.11, 25.51, 20.23,  7.71, 32.26, 0,     19.38, 14.75],
    [11.95, 11.53,  9.84, 21.17, 33.69, 19.38, 0,     18.26],
    [10.08, 21.86, 24.95, 21.38, 24.62, 14.75, 18.26, 0],
], dtype=float)

FROZEN_TIME = np.array([
    [0,    0.27, 0.44, 0.29, 0.52, 0.14, 0.28, 0.22],
    [0.27, 0,    0.38, 0.55, 0.79, 0.42, 0.28, 0.47],
    [0.44, 0.38, 0,    0.44, 0.92, 0.43, 0.23, 0.57],
    [0.29, 0.55, 0.44, 0,    0.79, 0.15, 0.51, 0.43],
    [0.52, 0.79, 0.92, 0.79, 0,    0.63, 0.75, 0.56],
    [0.14, 0.42, 0.43, 0.15, 0.63, 0,    0.35, 0.28],
    [0.28, 0.28, 0.23, 0.51, 0.75, 0.35, 0,    0.41],
    [0.22, 0.47, 0.57, 0.43, 0.56, 0.28, 0.41, 0],
], dtype=float)

FROZEN_ALLOWED = np.ones((8, 8), dtype=int)
np.fill_diagonal(FROZEN_ALLOWED, 0)

if os.path.exists(DATA_FILE):
    print("Loading:", DATA_FILE)

    nodes_df = pd.read_excel(DATA_FILE, sheet_name="Nodes_Demand")
    DIST = pd.read_excel(DATA_FILE, sheet_name="Distance_Matrix", index_col=0).astype(float)
    TIME = pd.read_excel(DATA_FILE, sheet_name="Travel_Time_Matrix", index_col=0).astype(float)
    ROUTE_ALLOWED = pd.read_excel(DATA_FILE, sheet_name="Route_Allowed", index_col=0).astype(int)
    parameters_df = pd.read_excel(DATA_FILE, sheet_name="Parameters")

    parameter_map = dict(zip(parameters_df["Parameter"], parameters_df["Value"]))

    factory_rows = nodes_df[nodes_df["Type"].str.lower() == "factory"]
    DEMAND = dict(zip(factory_rows["Location"], factory_rows["Demand_tonnes"].astype(float)))

    DEPOT_STOCK = float(parameter_map["Depot stock"])
    TRUCK_CAPACITY = float(parameter_map["Truck capacity"])
    TRANSPORT_RATE = float(parameter_map["Transport rate"])
    MAX_ROUTE_TIME = float(parameter_map["Maximum route duration"])
    DISPATCH_DAYS = int(parameter_map["Dispatch period"])

else:
    print("Excel file not found. Using embedded frozen QSOLVE dataset.")

    DEMAND = FROZEN_DEMAND.copy()
    DIST = pd.DataFrame(FROZEN_DISTANCE, index=LABELS, columns=LABELS)
    TIME = pd.DataFrame(FROZEN_TIME, index=LABELS, columns=LABELS)
    ROUTE_ALLOWED = pd.DataFrame(FROZEN_ALLOWED, index=LABELS, columns=LABELS)

    DEPOT_STOCK = 410.0
    TRUCK_CAPACITY = 10.0
    TRANSPORT_RATE = 16.0
    MAX_ROUTE_TIME = 8.0
    DISPATCH_DAYS = 10

TOTAL_DEMAND = sum(DEMAND.values())
MIN_TRIPS = math.ceil(TOTAL_DEMAND / TRUCK_CAPACITY)

print("\nTotal demand:", TOTAL_DEMAND)
print("Depot stock:", DEPOT_STOCK)
print("Truck capacity:", TRUCK_CAPACITY)
print("Transport rate:", TRANSPORT_RATE)
print("Maximum route time:", MAX_ROUTE_TIME)
print("Minimum theoretical trips:", MIN_TRIPS)

Loading: QSOLVE_Quantum_Ready_Dataset.xlsx

Total demand: 233.7
Depot stock: 410.0
Truck capacity: 10.0
Transport rate: 16.0
Maximum route time: 8.0
Minimum theoretical trips: 24


In [5]:
assert set(DEMAND) == set(FACTORIES)
assert abs(TOTAL_DEMAND - 233.700) < 1e-6
assert abs(DEPOT_STOCK - 410.0) < 1e-6
assert abs(TRUCK_CAPACITY - 10.0) < 1e-6
assert abs(TRANSPORT_RATE - 16.0) < 1e-6
assert abs(MAX_ROUTE_TIME - 8.0) < 1e-6

assert list(DIST.index) == LABELS
assert list(DIST.columns) == LABELS
assert list(TIME.index) == LABELS
assert list(TIME.columns) == LABELS

print("✓ Frozen dataset validated.")

display(pd.DataFrame({
    "Factory": FACTORIES,
    "Demand_tonnes": [DEMAND[f] for f in FACTORIES]
}))

display(DIST)
display(TIME)

✓ Frozen dataset validated.


,Factory,Demand_tonnes
0,Ragati,28.380
1,Nguguini,73.092
2,Kiangai,36.489
3,Thunguri,37.170
4,Kianjega,24.522
5,Chema,10.848
6,Mukangu,23.199


,Karatina,Ragati,Nguguini,Kiangai,Thunguri,Kianjega,Chema,Mukangu
From/To,,,,,,,,
Karatina,0.00,11.82,18.92,20.02,23.13,13.11,11.95,10.08
Ragati,11.82,0.00,15.86,29.64,34.81,25.51,11.53,21.86
Nguguini,18.92,15.86,0.00,19.08,40.65,20.23,9.84,24.95
Kiangai,20.02,29.64,19.08,0.00,38.90,7.71,21.17,21.38
Thunguri,23.13,34.81,40.65,38.90,0.00,32.26,33.69,24.62
Kianjega,13.11,25.51,20.23,7.71,32.26,0.00,19.38,14.75
Chema,11.95,11.53,9.84,21.17,33.69,19.38,0.00,18.26
Mukangu,10.08,21.86,24.95,21.38,24.62,14.75,18.26,0.00


,Karatina,Ragati,Nguguini,Kiangai,Thunguri,Kianjega,Chema,Mukangu
From/To,,,,,,,,
Karatina,0.00,0.27,0.44,0.29,0.52,0.14,0.28,0.22
Ragati,0.27,0.00,0.38,0.55,0.79,0.42,0.28,0.47
Nguguini,0.44,0.38,0.00,0.44,0.92,0.43,0.23,0.57
Kiangai,0.29,0.55,0.44,0.00,0.79,0.15,0.51,0.43
Thunguri,0.52,0.79,0.92,0.79,0.00,0.63,0.75,0.56
Kianjega,0.14,0.42,0.43,0.15,0.63,0.00,0.35,0.28
Chema,0.28,0.28,0.23,0.51,0.75,0.35,0.00,0.41
Mukangu,0.22,0.47,0.57,0.43,0.56,0.28,0.41,0.00


## 3. Classical capacity decomposition and multi-factory grouping

In [6]:
def split_full_and_residual_demand(demand, capacity):
    single_factory_trips = []
    residual_demand = {}

    for factory, qty in demand.items():
        full_trips = int(math.floor((qty + EPS) / capacity))

        for _ in range(full_trips):
            single_factory_trips.append({factory: capacity})

        remainder = qty - full_trips * capacity
        residual_demand[factory] = round(max(0.0, remainder), 3)

    return single_factory_trips, residual_demand


single_trips, residual = split_full_and_residual_demand(
    DEMAND,
    TRUCK_CAPACITY
)

print("Full 10-t single-factory trips:", len(single_trips))
print("Residual quantities:")

for factory, qty in residual.items():
    print(f"  {factory:<12}: {qty:>7.3f} t")

print("Residual total:", round(sum(residual.values()), 3), "t")

Full 10-t single-factory trips: 20
Residual quantities:
  Ragati      :   8.380 t
  Nguguini    :   3.092 t
  Kiangai     :   6.489 t
  Thunguri    :   7.170 t
  Kianjega    :   4.522 t
  Chema       :   0.848 t
  Mukangu     :   3.199 t
Residual total: 33.7 t


In [7]:
def create_multifactory_groups(
    residual_demand,
    distance_matrix,
    capacity=10.0,
    max_factories=4
):
    remaining = residual_demand.copy()
    groups = []

    while sum(remaining.values()) > EPS:
        active = [f for f, qty in remaining.items() if qty > EPS]

        if not active:
            break

        current = max(active, key=lambda f: remaining[f])
        remaining_capacity = capacity
        trip = {}

        while remaining_capacity > EPS and len(trip) < max_factories:
            if remaining[current] > EPS:
                delivery = min(remaining_capacity, remaining[current])
                trip[current] = trip.get(current, 0.0) + delivery
                remaining[current] -= delivery
                remaining_capacity -= delivery

            if remaining_capacity <= EPS:
                break

            candidates = [
                f for f, qty in remaining.items()
                if qty > EPS and f not in trip
            ]

            if not candidates:
                break

            current = min(
                candidates,
                key=lambda f: float(distance_matrix.loc[current, f])
            )

        groups.append({
            factory: round(qty, 3)
            for factory, qty in trip.items()
        })

    return groups


quantum_trip_groups = create_multifactory_groups(
    residual,
    DIST,
    capacity=TRUCK_CAPACITY,
    max_factories=MAX_QUANTUM_FACTORIES
)

print("Residual quantum groups:", len(quantum_trip_groups))

for idx, group in enumerate(quantum_trip_groups, 1):
    print(
        f"Group {idx}: {group} | "
        f"load={sum(group.values()):.3f} t | "
        f"factories={len(group)} | "
        f"qubits={len(group)**2}"
    )

print("\nRecombined trip count:", len(single_trips) + len(quantum_trip_groups))

Residual quantum groups: 4
Group 1: {'Ragati': 8.38, 'Chema': 0.848, 'Nguguini': 0.772} | load=10.000 t | factories=3 | qubits=9
Group 2: {'Thunguri': 7.17, 'Mukangu': 2.83} | load=10.000 t | factories=2 | qubits=4
Group 3: {'Kiangai': 6.489, 'Kianjega': 3.511} | load=10.000 t | factories=2 | qubits=4
Group 4: {'Nguguini': 2.32, 'Kianjega': 1.011, 'Mukangu': 0.369} | load=3.700 t | factories=3 | qubits=9

Recombined trip count: 24


## 4. Exact load-dependent route objective

\[
C = 16\sum_{(i,j)} d_{ij}L_{ij}
\]

where \(L_{ij}\) is the fertilizer carried on the road leg.

In [8]:
def route_metrics(order, deliveries):
    load = float(sum(deliveries.values()))
    initial_load = load
    current = DEPOT

    total_distance = 0.0
    total_time = 0.0
    total_tonne_km = 0.0
    total_cost = 0.0
    legs = []

    for factory in order:
        if int(ROUTE_ALLOWED.loc[current, factory]) != 1:
            return None

        d = float(DIST.loc[current, factory])
        t = float(TIME.loc[current, factory])

        leg_tonne_km = load * d
        leg_cost = TRANSPORT_RATE * leg_tonne_km
        delivered = float(deliveries[factory])

        legs.append({
            "From": current,
            "To": factory,
            "Load_tonnes": load,
            "Distance_km": d,
            "Time_hours": t,
            "Delivered_tonnes": delivered,
            "Tonne_km": leg_tonne_km,
            "Cost_KSh": leg_cost,
        })

        total_distance += d
        total_time += t
        total_tonne_km += leg_tonne_km
        total_cost += leg_cost

        load -= delivered
        current = factory

    if order:
        if int(ROUTE_ALLOWED.loc[current, DEPOT]) != 1:
            return None

        d = float(DIST.loc[current, DEPOT])
        t = float(TIME.loc[current, DEPOT])

        leg_tonne_km = load * d
        leg_cost = TRANSPORT_RATE * leg_tonne_km

        legs.append({
            "From": current,
            "To": DEPOT,
            "Load_tonnes": load,
            "Distance_km": d,
            "Time_hours": t,
            "Delivered_tonnes": 0.0,
            "Tonne_km": leg_tonne_km,
            "Cost_KSh": leg_cost,
        })

        total_distance += d
        total_time += t
        total_tonne_km += leg_tonne_km
        total_cost += leg_cost

    return {
        "order": list(order),
        "initial_load": initial_load,
        "distance_km": total_distance,
        "time_hours": total_time,
        "tonne_km": total_tonne_km,
        "cost_ksh": total_cost,
        "remaining_load": load,
        "legs": legs,
    }


def exact_route_benchmark(deliveries):
    best = None

    for order in permutations(deliveries.keys()):
        metrics = route_metrics(order, deliveries)

        if metrics is None:
            continue

        if metrics["time_hours"] > MAX_ROUTE_TIME + EPS:
            continue

        if best is None or metrics["cost_ksh"] < best["cost_ksh"]:
            best = metrics

    return best

## 5. Build the CUBO

In [9]:
def add_polynomial_term(polynomial, coefficient, variables):
    # Binary x satisfies x^2 = x, so repeated indices collapse.
    key = tuple(sorted(set(variables)))
    polynomial[key] += float(coefficient)


def build_route_cubo(deliveries, penalty=50.0, cost_scale=1000.0):
    cluster = list(deliveries.keys())
    m = len(cluster)

    variable_index = {}
    counter = 0

    for factory in cluster:
        for position in range(m):
            variable_index[(factory, position)] = counter
            counter += 1

    n_qubits = counter
    polynomial = defaultdict(float)
    total_load = float(sum(deliveries.values()))

    # Karatina -> first factory.
    for factory in cluster:
        variable = variable_index[(factory, 0)]
        d = float(DIST.loc[DEPOT, factory])

        add_polynomial_term(
            polynomial,
            TRANSPORT_RATE * total_load * d / cost_scale,
            [variable]
        )

    # Consecutive factory transitions.
    for position in range(1, m):
        previous_position = position - 1

        for factory_i in cluster:
            for factory_j in cluster:
                if factory_i == factory_j:
                    continue

                xi = variable_index[(factory_i, previous_position)]
                xj = variable_index[(factory_j, position)]
                d = float(DIST.loc[factory_i, factory_j])

                # Base load.
                add_polynomial_term(
                    polynomial,
                    TRANSPORT_RATE * d * total_load / cost_scale,
                    [xi, xj]
                )

                # Remove quantities delivered at earlier positions.
                for earlier_position in range(position):
                    for delivered_factory in cluster:
                        xh = variable_index[(delivered_factory, earlier_position)]
                        qty = float(deliveries[delivered_factory])

                        add_polynomial_term(
                            polynomial,
                            -TRANSPORT_RATE * d * qty / cost_scale,
                            [xi, xj, xh]
                        )

    # Each factory occurs once.
    for factory in cluster:
        variables = [variable_index[(factory, p)] for p in range(m)]

        add_polynomial_term(polynomial, penalty, [])

        for variable in variables:
            add_polynomial_term(polynomial, -penalty, [variable])

        for a, b in combinations(variables, 2):
            add_polynomial_term(polynomial, 2 * penalty, [a, b])

    # Each position contains one factory.
    for position in range(m):
        variables = [variable_index[(factory, position)] for factory in cluster]

        add_polynomial_term(polynomial, penalty, [])

        for variable in variables:
            add_polynomial_term(polynomial, -penalty, [variable])

        for a, b in combinations(variables, 2):
            add_polynomial_term(polynomial, 2 * penalty, [a, b])

    return {
        "cluster": cluster,
        "variable_index": variable_index,
        "n_qubits": n_qubits,
        "polynomial": dict(polynomial),
        "cost_scale": cost_scale,
        "penalty": penalty,
    }


def evaluate_binary_polynomial(polynomial, bits):
    value = 0.0

    for variables, coefficient in polynomial.items():
        product = 1.0

        for variable in variables:
            product *= bits[variable]

        value += coefficient * product

    return value

In [9]:
# Verify the CUBO reproduces exact route cost for every feasible permutation.

for idx, deliveries in enumerate(quantum_trip_groups, 1):
    model = build_route_cubo(deliveries)

    max_error = 0.0

    for order in permutations(model["cluster"]):
        bits = [0] * model["n_qubits"]

        for position, factory in enumerate(order):
            bits[model["variable_index"][(factory, position)]] = 1

        cubo_cost = (
            evaluate_binary_polynomial(model["polynomial"], bits)
            * model["cost_scale"]
        )

        exact_cost = route_metrics(order, deliveries)["cost_ksh"]
        max_error = max(max_error, abs(cubo_cost - exact_cost))

    print(f"Group {idx}: maximum cost mismatch = {max_error:.12f} KSh")
    assert max_error < 1e-6

print("✓ CUBO/exact objective consistency test passed.")

Group 1: maximum cost mismatch = 0.000000000029 KSh
Group 2: maximum cost mismatch = 0.000000000006 KSh
Group 3: maximum cost mismatch = 0.000000000017 KSh
Group 4: maximum cost mismatch = 0.000000000019 KSh
✓ CUBO/exact objective consistency test passed.


## 6. CUBO → Ising

Use:

\[
x_i=\frac{1-Z_i}{2}.
\]

This produces a diagonal Hamiltonian containing \(Z\), \(ZZ\), and \(ZZZ\) terms.

In [10]:
def cubo_to_ising(polynomial, n_qubits):
    ising_terms = defaultdict(float)

    for variables, coefficient in polynomial.items():
        degree = len(variables)

        if degree == 0:
            ising_terms[tuple()] += coefficient
            continue

        for subset_size in range(degree + 1):
            for subset in combinations(variables, subset_size):
                sign = -1 if subset_size % 2 else 1
                term_coefficient = coefficient * sign / (2 ** degree)

                key = tuple(sorted(subset))
                ising_terms[key] += term_coefficient

    labels = []
    coefficients = []

    for qubits, coefficient in ising_terms.items():
        if abs(coefficient) < 1e-12:
            continue

        label = ["I"] * n_qubits

        for q in qubits:
            label[n_qubits - 1 - q] = "Z"

        labels.append("".join(label))
        coefficients.append(complex(coefficient))

    return SparsePauliOp(
        labels,
        coeffs=np.asarray(coefficients, dtype=complex)
    ).simplify()

## 7. Decode QAOA samples

In [11]:
def bits_from_state_key(state, n_qubits):
    if isinstance(state, str):
        bitstring = state.replace(" ", "").zfill(n_qubits)
    else:
        bitstring = format(int(state), f"0{n_qubits}b")

    return [int(bit) for bit in bitstring[::-1]]


def assignment_is_feasible(bits, cluster, variable_index):
    m = len(cluster)

    for factory in cluster:
        if sum(
            bits[variable_index[(factory, p)]]
            for p in range(m)
        ) != 1:
            return False

    for position in range(m):
        if sum(
            bits[variable_index[(factory, position)]]
            for factory in cluster
        ) != 1:
            return False

    return True


def decode_order(bits, cluster, variable_index):
    m = len(cluster)
    order = []

    for position in range(m):
        selected = [
            factory
            for factory in cluster
            if bits[variable_index[(factory, position)]] == 1
        ]

        if len(selected) != 1:
            return None

        order.append(selected[0])

    return order


def extract_feasible_samples(
    qaoa_result,
    cluster,
    variable_index,
    n_qubits,
    deliveries
):
    distribution = qaoa_result.eigenstate

    if distribution is None:
        return []

    candidates = []

    for state, probability in distribution.items():
        bits = bits_from_state_key(state, n_qubits)

        if not assignment_is_feasible(bits, cluster, variable_index):
            continue

        order = decode_order(bits, cluster, variable_index)

        if order is None:
            continue

        metrics = route_metrics(order, deliveries)

        if metrics is None:
            continue

        if metrics["time_hours"] > MAX_ROUTE_TIME + EPS:
            continue

        candidates.append({
            "state": state,
            "probability": float(probability),
            "order": order,
            "metrics": metrics,
        })

    return candidates

## 8. QAOA solver

In [12]:
def solve_trip_with_qaoa(
    deliveries,
    reps=QAOA_REPS,
    shots=SHOTS,
    maxiter=MAXITER,
    seeds=RANDOM_SEEDS,
    cvar_alpha=CVAR_ALPHA,
):
    cluster = list(deliveries.keys())

    if len(cluster) == 1:
        metrics = route_metrics(cluster, deliveries)

        return {
            "method": "Direct single factory",
            "order": cluster,
            "metrics": metrics,
            "qubits": 0,
            "probability": 1.0,
            "quantum_energy": None,
            "runtime_seconds": 0.0,
            "exact_order": cluster,
            "exact_cost": metrics["cost_ksh"],
            "approximation_ratio": 1.0,
            "fallback": False,
            "qaoa_result": None,
            "hamiltonian": None,
        }

    model = build_route_cubo(deliveries)
    variable_index = model["variable_index"]
    n_qubits = model["n_qubits"]

    if n_qubits > MAX_QUANTUM_FACTORIES ** 2:
        raise ValueError(f"Quantum group too large: {n_qubits} qubits.")

    hamiltonian = cubo_to_ising(model["polynomial"], n_qubits)
    exact = exact_route_benchmark(deliveries)

    if exact is None:
        raise RuntimeError("No time-feasible exact route exists for this group.")

    print("\n" + "=" * 72)
    print("QAOA MULTI-FACTORY SUBPROBLEM")
    print("=" * 72)
    print("Deliveries:", deliveries)
    print("Factories:", cluster)
    print("Qubits:", n_qubits)
    print("CUBO terms:", len(model["polynomial"]))
    print("Ising Pauli terms:", len(hamiltonian))

    best_quantum = None

    for restart, seed in enumerate(seeds, start=1):
        print(f"\nRestart {restart}/{len(seeds)} | seed={seed}")

        algorithm_globals.random_seed = seed

        sampler = StatevectorSampler(
            default_shots=shots,
            seed=seed
        )

        optimizer = COBYLA(maxiter=maxiter)

        rng = np.random.default_rng(seed)
        initial_point = rng.uniform(
            low=0.0,
            high=np.pi,
            size=2 * reps
        )

        qaoa = QAOA(
            sampler=sampler,
            optimizer=optimizer,
            reps=reps,
            initial_point=initial_point,
            aggregation=cvar_alpha
        )

        start = time.perf_counter()
        result = qaoa.compute_minimum_eigenvalue(hamiltonian)
        elapsed = time.perf_counter() - start

        candidates = extract_feasible_samples(
            result,
            cluster,
            variable_index,
            n_qubits,
            deliveries
        )

        print("Feasible sampled routes:", len(candidates))

        if not candidates:
            continue

        current_best = min(
            candidates,
            key=lambda item: (
                item["metrics"]["cost_ksh"],
                -item["probability"]
            )
        )

        current_best["runtime_seconds"] = elapsed
        current_best["quantum_energy"] = float(np.real(result.eigenvalue))
        current_best["qaoa_result"] = result

        print("Best QAOA route:", current_best["order"])
        print("QAOA cost: KSh", round(current_best["metrics"]["cost_ksh"], 2))
        print("Probability:", round(current_best["probability"], 6))

        if (
            best_quantum is None
            or current_best["metrics"]["cost_ksh"]
            < best_quantum["metrics"]["cost_ksh"]
        ):
            best_quantum = current_best

    fallback = False

    if best_quantum is None:
        print("\nWARNING: no feasible QAOA sample. Exact small-route fallback used.")
        fallback = True

        best_quantum = {
            "order": exact["order"],
            "metrics": exact,
            "probability": 0.0,
            "runtime_seconds": np.nan,
            "quantum_energy": np.nan,
            "qaoa_result": None,
        }

    quantum_cost = best_quantum["metrics"]["cost_ksh"]

    approximation_ratio = (
        quantum_cost / exact["cost_ksh"]
        if exact["cost_ksh"] > EPS
        else 1.0
    )

    print("Exact route:", exact["order"])
    print("Exact cost: KSh", round(exact["cost_ksh"], 2))
    print("Approximation ratio:", round(approximation_ratio, 6))

    return {
        "method": "CUBO-QAOA",
        "order": best_quantum["order"],
        "metrics": best_quantum["metrics"],
        "qubits": n_qubits,
        "probability": best_quantum["probability"],
        "quantum_energy": best_quantum["quantum_energy"],
        "runtime_seconds": best_quantum["runtime_seconds"],
        "exact_order": exact["order"],
        "exact_cost": exact["cost_ksh"],
        "approximation_ratio": approximation_ratio,
        "fallback": fallback,
        "qaoa_result": best_quantum["qaoa_result"],
        "hamiltonian": hamiltonian,
    }

## 9. Test one quantum group

In [13]:
test_group = quantum_trip_groups[0]

test_result = solve_trip_with_qaoa(test_group)

print("\nTEST QUANTUM RESULT")
print("Route:", DEPOT, "->", " -> ".join(test_result["order"]), "->", DEPOT)
print("QAOA cost: KSh", round(test_result["metrics"]["cost_ksh"], 2))
print("Exact cost: KSh", round(test_result["exact_cost"], 2))
print("Approximation ratio:", round(test_result["approximation_ratio"], 6))
print("Qubits:", test_result["qubits"])


QAOA MULTI-FACTORY SUBPROBLEM
Deliveries: {'Ragati': 8.38, 'Chema': 0.848, 'Nguguini': 0.772}
Factories: ['Ragati', 'Chema', 'Nguguini']
Qubits: 9
CUBO terms: 76
Ising Pauli terms: 82

Restart 1/1 | seed=42


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Feasible sampled routes: 6
Best QAOA route: ['Ragati', 'Chema', 'Nguguini']
QAOA cost: KSh 2311.6
Probability: 0.007324
Exact route: ['Ragati', 'Chema', 'Nguguini']
Exact cost: KSh 2311.6
Approximation ratio: 1.0

TEST QUANTUM RESULT
Route: Karatina -> Ragati -> Chema -> Nguguini -> Karatina
QAOA cost: KSh 2311.6
Exact cost: KSh 2311.6
Approximation ratio: 1.0
Qubits: 9


## 10. Solve the complete hybrid system

In [ ]:
all_route_results = []
quantum_subproblem_results = []
trip_id = 1

# Full single-factory 10-t trips.
for deliveries in single_trips:
    factory = list(deliveries.keys())[0]
    metrics = route_metrics([factory], deliveries)

    all_route_results.append({
        "Trip": trip_id,
        "Method": "Classical fixed full load",
        "Route": f"{DEPOT} -> {factory} -> {DEPOT}",
        "Deliveries": f"{factory}: {deliveries[factory]:.3f} t",
        "Delivered_tonnes": sum(deliveries.values()),
        "Distance_km": metrics["distance_km"],
        "Time_hours": metrics["time_hours"],
        "Tonne_km": metrics["tonne_km"],
        "Cost_KSh": metrics["cost_ksh"],
        "Qubits": 0,
        "QAOA_probability": np.nan,
        "Approximation_ratio": np.nan,
        "Quantum_fallback": False,
    })

    trip_id += 1


# Quantum residual multi-factory trips.
for group_number, deliveries in enumerate(quantum_trip_groups, start=1):
    print("\n" + "#" * 80)
    print(f"SOLVING QUANTUM GROUP {group_number}/{len(quantum_trip_groups)}")
    print("#" * 80)

    result = solve_trip_with_qaoa(deliveries)
    metrics = result["metrics"]
    order = result["order"]

    route_string = DEPOT + " -> " + " -> ".join(order) + " -> " + DEPOT

    delivery_string = "; ".join(
        f"{factory}: {deliveries[factory]:.3f} t"
        for factory in deliveries
    )

    all_route_results.append({
        "Trip": trip_id,
        "Method": result["method"],
        "Route": route_string,
        "Deliveries": delivery_string,
        "Delivered_tonnes": sum(deliveries.values()),
        "Distance_km": metrics["distance_km"],
        "Time_hours": metrics["time_hours"],
        "Tonne_km": metrics["tonne_km"],
        "Cost_KSh": metrics["cost_ksh"],
        "Qubits": result["qubits"],
        "QAOA_probability": result["probability"],
        "Approximation_ratio": result["approximation_ratio"],
        "Quantum_fallback": result["fallback"],
    })

    quantum_subproblem_results.append({
        "Trip": trip_id,
        "Deliveries": deliveries.copy(),
        "QAOA_Order": order,
        "QAOA_Cost": metrics["cost_ksh"],
        "Exact_Order": result["exact_order"],
        "Exact_Cost": result["exact_cost"],
        "Approximation_Ratio": result["approximation_ratio"],
        "Qubits": result["qubits"],
        "Runtime_seconds": result["runtime_seconds"],
        "Fallback": result["fallback"],
        "QAOA_Result_Object": result["qaoa_result"],
        "Hamiltonian": result["hamiltonian"],
    })

    trip_id += 1

routes_df = pd.DataFrame(all_route_results)
display(routes_df)


################################################################################
SOLVING QUANTUM GROUP 1/4
################################################################################

QAOA MULTI-FACTORY SUBPROBLEM
Deliveries: {'Ragati': 8.38, 'Chema': 0.848, 'Nguguini': 0.772}
Factories: ['Ragati', 'Chema', 'Nguguini']
Qubits: 9
CUBO terms: 76
Ising Pauli terms: 82

Restart 1/1 | seed=42


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Feasible sampled routes: 6
Best QAOA route: ['Ragati', 'Chema', 'Nguguini']
QAOA cost: KSh 2311.6
Probability: 0.007324
Exact route: ['Ragati', 'Chema', 'Nguguini']
Exact cost: KSh 2311.6
Approximation ratio: 1.0

################################################################################
SOLVING QUANTUM GROUP 2/4
################################################################################

QAOA MULTI-FACTORY SUBPROBLEM
Deliveries: {'Thunguri': 7.17, 'Mukangu': 2.83}
Factories: ['Thunguri', 'Mukangu']
Qubits: 4
CUBO terms: 13
Ising Pauli terms: 13

Restart 1/1 | seed=42


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Feasible sampled routes: 2
Best QAOA route: ['Mukangu', 'Thunguri']
QAOA cost: KSh 4437.21
Probability: 0.197266
Exact route: ['Mukangu', 'Thunguri']
Exact cost: KSh 4437.21
Approximation ratio: 1.0

################################################################################
SOLVING QUANTUM GROUP 3/4
################################################################################

QAOA MULTI-FACTORY SUBPROBLEM
Deliveries: {'Kiangai': 6.489, 'Kianjega': 3.511}
Factories: ['Kiangai', 'Kianjega']
Qubits: 4
CUBO terms: 13
Ising Pauli terms: 13

Restart 1/1 | seed=42


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Feasible sampled routes: 2
Best QAOA route: ['Kianjega', 'Kiangai']
QAOA cost: KSh 2898.08
Probability: 0.220215
Exact route: ['Kianjega', 'Kiangai']
Exact cost: KSh 2898.08
Approximation ratio: 1.0

################################################################################
SOLVING QUANTUM GROUP 4/4
################################################################################

QAOA MULTI-FACTORY SUBPROBLEM
Deliveries: {'Nguguini': 2.32, 'Kianjega': 1.011, 'Mukangu': 0.369}
Factories: ['Nguguini', 'Kianjega', 'Mukangu']
Qubits: 9
CUBO terms: 76
Ising Pauli terms: 82

Restart 1/1 | seed=42


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


## 11. Validate demand, capacity, stock, and route time

In [ ]:
delivered_by_factory = {factory: 0.0 for factory in FACTORIES}

for trip in single_trips:
    for factory, qty in trip.items():
        delivered_by_factory[factory] += qty

for trip in quantum_trip_groups:
    for factory, qty in trip.items():
        delivered_by_factory[factory] += qty

validation_rows = []

for factory in FACTORIES:
    demand_qty = DEMAND[factory]
    delivered_qty = delivered_by_factory[factory]
    unmet_qty = max(0.0, demand_qty - delivered_qty)

    validation_rows.append({
        "Factory": factory,
        "Demand_tonnes": demand_qty,
        "Delivered_tonnes": delivered_qty,
        "Unmet_tonnes": unmet_qty,
        "Fulfillment_%": delivered_qty / demand_qty * 100.0,
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

TOTAL_COST = routes_df["Cost_KSh"].sum()
TOTAL_DISTANCE = routes_df["Distance_km"].sum()
TOTAL_TONNE_KM = routes_df["Tonne_km"].sum()
TOTAL_DELIVERED = routes_df["Delivered_tonnes"].sum()
TOTAL_TRIPS = len(routes_df)
QUANTUM_TRIPS = int((routes_df["Method"] == "CUBO-QAOA").sum())
REMAINING_STOCK = DEPOT_STOCK - TOTAL_DELIVERED

print("=" * 80)
print("FINAL HYBRID QUANTUM RESULTS")
print("=" * 80)
print(f"Total demand          : {TOTAL_DEMAND:.3f} t")
print(f"Total delivered       : {TOTAL_DELIVERED:.3f} t")
print(f"Unmet demand          : {TOTAL_DEMAND - TOTAL_DELIVERED:.6f} t")
print(f"Total routes/trips    : {TOTAL_TRIPS}")
print(f"Quantum multi-routes  : {QUANTUM_TRIPS}")
print(f"Total distance        : {TOTAL_DISTANCE:.2f} km")
print(f"Total tonne-km        : {TOTAL_TONNE_KM:.2f}")
print(f"Transport cost        : KSh {TOTAL_COST:,.2f}")
print(f"Remaining depot stock : {REMAINING_STOCK:.3f} t")

print("\nCONSTRAINT CHECKS")
print("Demand satisfied:", abs(TOTAL_DELIVERED - TOTAL_DEMAND) < 1e-6)
print("Depot stock valid:", TOTAL_DELIVERED <= DEPOT_STOCK + 1e-6)
print("All routes <= 8 h:", bool((routes_df["Time_hours"] <= MAX_ROUTE_TIME + 1e-6).all()))
print("All trips <= 10 t:", bool((routes_df["Delivered_tonnes"] <= TRUCK_CAPACITY + 1e-6).all()))
print("Trip count >= theoretical minimum:", TOTAL_TRIPS >= MIN_TRIPS)

## 12. Quantum accuracy table

\[
\text{Approximation ratio}
=
\frac{C_{\mathrm{QAOA}}}{C_{\mathrm{exact}}}.
\]

A value of 1.0 means QAOA matched the exact optimum for that small multi-factory subproblem.

In [ ]:
quantum_accuracy_rows = []

for result in quantum_subproblem_results:
    quantum_accuracy_rows.append({
        "Trip": result["Trip"],
        "Qubits": result["Qubits"],
        "QAOA_Route": " -> ".join(result["QAOA_Order"]),
        "QAOA_Cost_KSh": result["QAOA_Cost"],
        "Exact_Route": " -> ".join(result["Exact_Order"]),
        "Exact_Cost_KSh": result["Exact_Cost"],
        "Approximation_Ratio": result["Approximation_Ratio"],
        "Runtime_seconds": result["Runtime_seconds"],
        "Fallback": result["Fallback"],
    })

quantum_accuracy_df = pd.DataFrame(quantum_accuracy_rows)
display(quantum_accuracy_df)

if len(quantum_accuracy_df):
    print("Mean approximation ratio:", quantum_accuracy_df["Approximation_Ratio"].mean())

## 13. Export quantum results to Excel

In [ ]:
OUTPUT_FILE = "QSOLVE_Hybrid_Quantum_CUBO_QAOA_Results.xlsx"

summary_df = pd.DataFrame({
    "Metric": [
        "Algorithm",
        "Total Demand (t)",
        "Total Delivered (t)",
        "Total Trips",
        "Quantum Multi-Factory Trips",
        "Total Distance (km)",
        "Total Tonne-km",
        "Total Transport Cost (KSh)",
        "Depot Stock (t)",
        "Remaining Stock (t)",
        "Truck Capacity (t)",
        "Max Route Time (h)",
        "QAOA reps",
        "Shots",
        "FAST_MODE",
    ],
    "Value": [
        "Hybrid CUBO-QAOA",
        TOTAL_DEMAND,
        TOTAL_DELIVERED,
        TOTAL_TRIPS,
        QUANTUM_TRIPS,
        TOTAL_DISTANCE,
        TOTAL_TONNE_KM,
        TOTAL_COST,
        DEPOT_STOCK,
        REMAINING_STOCK,
        TRUCK_CAPACITY,
        MAX_ROUTE_TIME,
        QAOA_REPS,
        SHOTS,
        FAST_MODE,
    ]
})

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    routes_df.to_excel(writer, sheet_name="All_Routes", index=False)
    validation_df.to_excel(writer, sheet_name="Factory_Validation", index=False)
    quantum_accuracy_df.to_excel(writer, sheet_name="Quantum_Benchmark", index=False)

print("Saved:", OUTPUT_FILE)

## 14. Optional qBraid QIR cloud-simulator cell

After at least one QAOA group completes locally, you can submit its optimized circuit to qBraid's QIR simulator `qbraid:qbraid:sim:qir-sv`.

The cell is commented out so the main notebook does not require qBraid cloud execution.

In [ ]:
# OPTIONAL qBRAID CLOUD EXECUTION
#
# from qbraid import QbraidProvider
#
# cloud_candidate = None
#
# for item in quantum_subproblem_results:
#     if item["QAOA_Result_Object"] is not None:
#         cloud_candidate = item
#         break
#
# if cloud_candidate is None:
#     print("No successful QAOA result is available yet.")
#
# else:
#     qaoa_result = cloud_candidate["QAOA_Result_Object"]
#
#     circuit = qaoa_result.optimal_circuit.assign_parameters(
#         qaoa_result.optimal_parameters
#     ).copy()
#
#     if circuit.num_clbits == 0:
#         circuit.measure_all()
#
#     print("Circuit qubits:", circuit.num_qubits)
#     print("Circuit depth:", circuit.depth())
#
#     provider = QbraidProvider()
#     device = provider.get_device("qbraid:qbraid:sim:qir-sv")
#
#     job = device.run(circuit, shots=2000)
#     print("Job:", job)
#
#     cloud_result = job.result()
#     counts = cloud_result.data.get_counts()
#
#     print("qBraid QIR counts:")
#     print(counts)

## 15. Benchmark progression

After the first successful run:

1. Set `FAST_MODE=False`.
2. Run all three seeds.
3. Compare QAOA \(p=1,2,3\).
4. Record qubits, Pauli terms, runtime, route cost, exact cost, and approximation ratio.
5. Build at least five increasing quantum problem instances.
6. Compare the quantum-hybrid results against the classical MILP using the same frozen dataset and objective.

In [ ]:
# ============================================================
# EXTRA CELL 1
# EXTRACT MULTI-FACTORY ROUTES
# ============================================================

def extract_factory_sequence(route_string):
    """
    Convert:
        Karatina -> Kianjega -> Kiangai -> Karatina

    into:
        ['Kianjega', 'Kiangai']
    """

    locations = [
        location.strip()
        for location in route_string.split("->")
    ]

    return [
        location
        for location in locations
        if location != DEPOT
    ]


# Add the factory sequence to every route
routes_df["Factory_Sequence"] = (
    routes_df["Route"]
    .apply(extract_factory_sequence)
)


# Count number of factories served/visited
routes_df["Number_of_Factories"] = (
    routes_df["Factory_Sequence"]
    .apply(len)
)


# Keep routes containing at least two factories
multi_factory_routes_df = (
    routes_df[
        routes_df["Number_of_Factories"] >= 2
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 100)
print("ALL MULTI-FACTORY ROUTES")
print("=" * 100)

display(
    multi_factory_routes_df[
        [
            "Trip",
            "Method",
            "Route",
            "Factory_Sequence",
            "Deliveries",
            "Delivered_tonnes",
            "Distance_km",
            "Time_hours",
            "Tonne_km",
            "Cost_KSh",
            "Qubits",
            "QAOA_probability",
            "Approximation_ratio"
        ]
    ]
)


print(
    "\nNumber of multi-factory routes:",
    len(multi_factory_routes_df)
)

In [20]:
# ============================================================
# EXTRA CELL 2
# EXTRACT ONLY QAOA MULTI-FACTORY ROUTES
# ============================================================

quantum_multi_factory_df = (
    routes_df[
        (routes_df["Method"] == "CUBO-QAOA")
        &
        (routes_df["Number_of_Factories"] >= 2)
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 100)
print("QAOA MULTI-FACTORY ROUTES")
print("=" * 100)


display(
    quantum_multi_factory_df[
        [
            "Trip",
            "Route",
            "Factory_Sequence",
            "Deliveries",
            "Delivered_tonnes",
            "Distance_km",
            "Time_hours",
            "Tonne_km",
            "Cost_KSh",
            "Qubits",
            "QAOA_probability",
            "Approximation_ratio",
            "Quantum_fallback"
        ]
    ]
)


print(
    "\nQuantum multi-factory routes:",
    len(quantum_multi_factory_df)
)

QAOA MULTI-FACTORY ROUTES


,Trip,Route,Factory_Sequence,Deliveries,Delivered_tonnes,Distance_km,Time_hours,Tonne_km,Cost_KSh,Qubits,QAOA_probability,Approximation_ratio,Quantum_fallback
0,21,Karatina -> Ragati -> Chema -> Nguguini -> Kar...,"[Ragati, Chema, Nguguini]",Ragati: 8.380 t; Chema: 0.848 t; Nguguini: 0.7...,10.0,52.11,1.22,144.47508,2311.60128,9,0.007324,1.0,False
1,22,Karatina -> Mukangu -> Thunguri -> Karatina,"[Mukangu, Thunguri]",Thunguri: 7.170 t; Mukangu: 2.830 t,10.0,57.83,1.30,277.32540,4437.20640,4,0.197266,1.0,False
2,23,Karatina -> Kianjega -> Kiangai -> Karatina,"[Kianjega, Kiangai]",Kiangai: 6.489 t; Kianjega: 3.511 t,10.0,40.84,0.58,181.13019,2898.08304,4,0.220215,1.0,False
3,24,Karatina -> Nguguini -> Kianjega -> Mukangu ->...,"[Nguguini, Kianjega, Mukangu]",Nguguini: 2.320 t; Kianjega: 1.011 t; Mukangu:...,3.7,63.98,1.37,103.36415,1653.82640,9,0.009277,1.0,False



Quantum multi-factory routes: 4


In [ ]:
# ============================================================
# EXTRA CELL 3
# PRINT MULTI-FACTORY ROUTES CLEARLY
# ============================================================

for _, row in quantum_multi_factory_df.iterrows():

    print("\n" + "=" * 80)

    print(
        f"QUANTUM TRIP {int(row['Trip'])}"
    )

    print("-" * 80)

    print(
        f"Route       : {row['Route']}"
    )

    print(
        f"Factories   : "
        f"{' -> '.join(row['Factory_Sequence'])}"
    )

    print(
        f"Deliveries  : {row['Deliveries']}"
    )

    print(
        f"Total load  : "
        f"{row['Delivered_tonnes']:.3f} tonnes"
    )

    print(
        f"Distance    : "
        f"{row['Distance_km']:.2f} km"
    )

    print(
        f"Travel time : "
        f"{row['Time_hours']:.2f} hours"
    )

    print(
        f"Tonne-km    : "
        f"{row['Tonne_km']:.3f}"
    )

    print(
        f"Cost        : "
        f"KSh {row['Cost_KSh']:,.2f}"
    )

    print(
        f"Qubits      : "
        f"{int(row['Qubits'])}"
    )

    print(
        f"QAOA probability : "
        f"{row['QAOA_probability']:.6f}"
    )

    print(
        f"Approximation ratio : "
        f"{row['Approximation_ratio']:.6f}"
    )

    print(
        f"Fallback used : "
        f"{row['Quantum_fallback']}"
    )

In [ ]:
# ============================================================
# EXTRA CELL 4
# LEG-BY-LEG DETAILS FOR QUANTUM MULTI-FACTORY ROUTES
# ============================================================

multi_factory_leg_rows = []


for result in quantum_subproblem_results:

    trip_id = result["Trip"]

    deliveries = result["Deliveries"]

    order = result["QAOA_Order"]


    # Recalculate exact route details
    metrics = route_metrics(
        order,
        deliveries
    )


    for leg_number, leg in enumerate(
        metrics["legs"],
        start=1
    ):

        multi_factory_leg_rows.append({

            "Trip":
                trip_id,

            "Leg":
                leg_number,

            "From":
                leg["From"],

            "To":
                leg["To"],

            "Load_Before_Delivery_t":
                leg["Load_tonnes"],

            "Delivered_at_Destination_t":
                leg["Delivered_tonnes"],

            "Distance_km":
                leg["Distance_km"],

            "Travel_Time_hours":
                leg["Time_hours"],

            "Tonne_km":
                leg["Tonne_km"],

            "Segment_Cost_KSh":
                leg["Cost_KSh"]
        })


multi_factory_legs_df = pd.DataFrame(
    multi_factory_leg_rows
)


print("=" * 100)
print("LEG-BY-LEG QUANTUM MULTI-FACTORY ROUTES")
print("=" * 100)


display(
    multi_factory_legs_df
)

In [ ]:
# ============================================================
# EXTRA CELL 5
# DISPLAY LOAD PROGRESSION
# ============================================================

for trip_id in sorted(
    multi_factory_legs_df["Trip"].unique()
):

    trip_legs = (
        multi_factory_legs_df[
            multi_factory_legs_df["Trip"] == trip_id
        ]
    )


    print("\n" + "=" * 80)

    print(
        f"TRIP {trip_id} - LOAD PROGRESSION"
    )

    print("=" * 80)


    for _, leg in trip_legs.iterrows():

        print(
            f"{leg['From']:<12}"
            f" -> "
            f"{leg['To']:<12}"
            f" | Load = "
            f"{leg['Load_Before_Delivery_t']:.3f} t"
            f" | Delivered = "
            f"{leg['Delivered_at_Destination_t']:.3f} t"
            f" | Cost = "
            f"KSh {leg['Segment_Cost_KSh']:,.2f}"
        )

In [24]:
# ============================================================
# EXTRA CELL 6
# QAOA ROUTE VS EXACT ROUTE
# ============================================================

comparison_rows = []


for result in quantum_subproblem_results:

    comparison_rows.append({

        "Trip":
            result["Trip"],

        "Deliveries":
            "; ".join(
                f"{factory}: {qty:.3f} t"
                for factory, qty
                in result["Deliveries"].items()
            ),

        "QAOA_Route":
            (
                DEPOT
                + " -> "
                + " -> ".join(
                    result["QAOA_Order"]
                )
                + " -> "
                + DEPOT
            ),

        "QAOA_Cost_KSh":
            result["QAOA_Cost"],

        "Exact_Route":
            (
                DEPOT
                + " -> "
                + " -> ".join(
                    result["Exact_Order"]
                )
                + " -> "
                + DEPOT
            ),

        "Exact_Cost_KSh":
            result["Exact_Cost"],

        "Cost_Difference_KSh":
            (
                result["QAOA_Cost"]
                - result["Exact_Cost"]
            ),

        "Approximation_Ratio":
            result["Approximation_Ratio"],

        "Qubits":
            result["Qubits"],

        "Runtime_seconds":
            result["Runtime_seconds"],

        "Fallback":
            result["Fallback"]
    })


quantum_route_comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
)


print("=" * 100)
print("QAOA VS EXACT MULTI-FACTORY ROUTING")
print("=" * 100)


display(
    quantum_route_comparison_df
)

QAOA VS EXACT MULTI-FACTORY ROUTING


,Trip,Deliveries,QAOA_Route,QAOA_Cost_KSh,Exact_Route,Exact_Cost_KSh,Cost_Difference_KSh,Approximation_Ratio,Qubits,Runtime_seconds,Fallback
0,21,Ragati: 8.380 t; Chema: 0.848 t; Nguguini: 0.7...,Karatina -> Ragati -> Chema -> Nguguini -> Kar...,2311.60128,Karatina -> Ragati -> Chema -> Nguguini -> Kar...,2311.60128,0.0,1.0,9,89.236395,False
1,22,Thunguri: 7.170 t; Mukangu: 2.830 t,Karatina -> Mukangu -> Thunguri -> Karatina,4437.20640,Karatina -> Mukangu -> Thunguri -> Karatina,4437.20640,0.0,1.0,4,1.304024,False
2,23,Kiangai: 6.489 t; Kianjega: 3.511 t,Karatina -> Kianjega -> Kiangai -> Karatina,2898.08304,Karatina -> Kianjega -> Kiangai -> Karatina,2898.08304,0.0,1.0,4,1.196297,False
3,24,Nguguini: 2.320 t; Kianjega: 1.011 t; Mukangu:...,Karatina -> Nguguini -> Kianjega -> Mukangu ->...,1653.82640,Karatina -> Nguguini -> Kianjega -> Mukangu ->...,1653.82640,0.0,1.0,9,89.432880,False
